# `06-workflow.ipynb`

In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

llm = init_chat_model('gpt-4.1-mini')

## Prompt Chaining

- 매우 잘 정리된 업무 순서가 있을 경우 사용
- 이전 노드에서 처리한 내용을 `state` 에 담아 다음 노드로 전송

In [7]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# 그림 보는 용
from IPython.display import Image, display


# Graph State
class State(TypedDict):
    topic: str
    joke: str
    improved_joke: str
    final_joke: str


# Nodes
def generate_joke(state: State):
    msg = llm.invoke(f'주제 {state['topic']}에 관련된 짧은 농담 생성')
    return {'joke': msg.content}


def improve_joke(state: State):
    msg = llm.invoke(f'말장난을 추가해서 아래 농담을 더 재밌게 만들어보자.\n {state['joke']}')
    return {'improved_joke': msg.content}


def polish_joke(state: State):
    msg = llm.invoke(f'아래 농담을 쩔게 뒤틀어 보자. \n {state['improved_joke']}')
    return {'final_joke': msg.content}


# Router
def check_punchline(state: State):
    # ? 나 ! 없으면, 농담을 improve 하고, 있으면 그대로 진행
    if '?' in state['joke'] or '!' in state['joke']:
        return 'Pass'
    else:
        return 'Fail'
    

workflow = StateGraph(State)

# 조립
workflow.add_node(generate_joke)
workflow.add_node(improve_joke)
workflow.add_node(polish_joke)

workflow.add_edge(START, 'generate_joke')
workflow.add_conditional_edges(
    'generate_joke',
    check_punchline,
    {
        'Pass': END,
        'Fali': 'improve_joke'
    }
)
workflow.add_edge('improve_joke', 'polish_joke')
workflow.add_edge('polish_joke', END)

graph = workflow.compile()
graph

graph.invoke({'topic': '랭체인'})

{'topic': '랭체인',
 'joke': '랭체인 개발자: "내 코드가 체인처럼 단단해!"  \n친구: "그래? 디버깅할 때는 안 풀리던데?"'}

## Parallelization (병렬화) (비동기)
- 여러 node(llm)이 동시에 작업을 진행
- 뭐가 먼저 끝날지 알 수 없음
- 하위 태스크를 동시에 진행시켜서 속도 up
- 같은 태스크를 여러번 동시에 돌려서 신뢰성 확보

In [9]:
class State(TypedDict):
    topic: str
    joke: str
    story: str
    poem: str
    output: str


# Nodes
def generate_joke(state: State):
    msg = llm.invoke(f'Write a joke about {state['topic']}')
    return {'joke': msg.content}


def generate_story(state: State):
    msg = llm.invoke(f'Write a story about {state['topic']}')
    return {'story': msg.content}


def generate_poem(state: State):
    msg = llm.invoke(f'Write a poem about {state['topic']}')
    return {'poem': msg.content}


def aggregate(state: State):
    output = f"""농담, 이야기, 시
Joke: {state['joke']} `
Story: {state['story']}
Poem: {state['poem']}
"""
    return {'output': output}


workflow = StateGraph(State)
workflow.add_node(generate_joke)
workflow.add_node(generate_poem)
workflow.add_node(generate_story)
workflow.add_node(aggregate)

workflow.add_edge(START, 'generate_joke')
workflow.add_edge(START, 'generate_poem')
workflow.add_edge(START, 'generate_story')
workflow.add_edge('generate_joke', 'aggregate')
workflow.add_edge('generate_poem', 'aggregate')
workflow.add_edge('generate_story', 'aggregate')
workflow.add_edge('aggregate', END)

graph = workflow.compile()

graph.invoke({'topic': '봄'})

{'topic': '봄',
 'joke': 'Why did the flower bring a jacket to 봄 (spring)?\n\nBecause it heard the weather might be a little *budding* chilly! 🌸😄',
 'story': '봄, 그 따스하고 포근한 계절이 찾아왔다. 겨울 동안 얼어붙었던 대지는 서서히 녹아내리고, 온 세상은 색색의 꽃들로 가득 찼다.\n\n어느 작은 마을에 사는 소녀 하늘이는 봄을 가장 좋아했다. 겨우내 집 안에서만 지내던 하늘이는 따뜻한 햇살과 싱그러운 바람을 맞으며 마당에 나왔다. 그녀의 눈앞에는 노란 개나리와 분홍 벚꽃이 활짝 피어 있어, 마치 꿈속에 들어온 듯한 기분이 들었다.\n\n하늘이는 친구들과 함께 마을 근처 숲으로 나갔다. 숲 속에는 새들의 노래와 작은 개울물 소리가 어우러져 봄의 향기를 가득 품고 있었다. 아이들은 손에 손을 잡고 뛰어다니며 봄꽃을 꺾고, 나비를 쫓았다.\n\n그날 저녁, 하늘이는 창가에 앉아 따뜻한 차 한 잔을 마시며 속으로 다짐했다. “내년에도 꼭 봄을 기다릴 거야. 봄처럼 따뜻하고 밝게 살고 싶어.”\n\n봄은 그렇게 그녀의 마음속에도 희망과 새로운 시작을 심어주었다. 그리고 매년 봄이 올 때마다, 하늘이는 다시 한 번 더 세상의 아름다움을 느끼며 웃음을 지었다.',
 'poem': '봄이 다가와 속삭이네,  \n살랑이는 바람에 꽃잎 춤추고,  \n햇살은 부드럽게 땅을 감싸니,  \n새싹은 희망의 노래를 부르네.\n\n차가운 겨울은 저 멀리 물러가고,  \n푸른 하늘 아래 세상은 깨어나,  \n사랑과 꿈을 심는 계절,  \n봄빛 속에 마음도 피어나네.',
 'output': '농담, 이야기, 시\nJoke: Why did the flower bring a jacket to 봄 (spring)?\n\nBecause it heard the weather might be a little *budding* chilly! 🌸😄 `\nStory: 봄, 그 따